# Nieh–Yan identity and the T/τ dictionary
# NY恒等式とT/τ対応の独立計算ノート

This optional companion to Paper 01 v3 reproduces the local calculations in
Section 2 and Appendices A–B. The paper contains the definitions, claims and
arguments; this notebook lets the reader inspect the algebra independently.
The code uses only SymPy and Python's standard library. It does not import the
paper's engine or precomputed results. Run the cells in order with a fresh kernel.

Paper 01 v3の第2章・付録A–Bに対応する任意の補足資料です。主張と論証は論文本体に
置き、このノートでは局所的な代数計算を独立に追います。使用するのはSymPyとPythonの
標準ライブラリだけです。論文の計算エンジンや保存済みの結果を読み込みません。
新しいkernelで上から順に実行してください。

Reading guides: [English](README.md) / [日本語](README_ja.md).

## 1. Definitions and scope / 定義と範囲

Use an orthonormal Euclidean coframe, indices $a,b,c=0,1,2,3$, spatial indices
$i,j,k=0,1,2$, and orientation $\varepsilon_{0123}=+1$. Fix

$$
\begin{aligned}
de^a&=\frac12C^a{}_{bc}e^b\wedge e^c,&
\omega^a{}_b&=\Gamma^a{}_{bc}e^c,\\
T^a&=de^a+\omega^a{}_b\wedge e^b
    =\frac12T^a{}_{bc}e^b\wedge e^c,&
R^a{}_b&=d\omega^a{}_b+\omega^a{}_c\wedge\omega^c{}_b
    =\frac12R^a{}_{bcd}e^c\wedge e^d.
\end{aligned}
$$

The last index of $\Gamma$ is the differentiation direction.
$s>0$ is a fixed coframe scale; $V,\eta$ are arbitrary real constants.
All coefficients are constant along the frame. In particular, the exterior
derivative below differentiates basis forms, not these parameters. Differentiating
a reduced potential with respect to $s$ is a different operation.

正規直交Euclidean coframeと上式の向き・添字順序を用います。$\Gamma$の最後の添字が
微分方向です。$s>0$、実数$V,\eta$は各固定幾何でframe位置によらない定数であり、
以下の外微分は基底形式を微分します。縮約ポテンシャルの$s$微分とは区別します。

In [ ]:
import sympy as sp
from itertools import combinations, permutations, product

indices = range(4)
s = sp.Symbol("s", positive=True)
V, eta = sp.symbols("V eta", real=True)
e = [{(a,): sp.S.One} for a in indices]
volume_indices = (0, 1, 2, 3)

def simplify_form(form):
    result = {}
    for basis, value in form.items():
        value = sp.factor(value)
        if value != 0:
            result[basis] = value
    return result

def add_forms(*forms):
    result = {}
    for form in forms:
        for basis, value in form.items():
            result[basis] = result.get(basis, sp.S.Zero) + value
    return simplify_form(result)

def multiply(value, form):
    return simplify_form({basis: value * x for basis, x in form.items()})

def permutation_sign(sequence):
    return (-1) ** sum(sequence[i] > sequence[j]
                      for i in range(len(sequence)) for j in range(i + 1, len(sequence)))

def wedge(left, right):
    result = {}
    for a, x in left.items():
        for b, y in right.items():
            sequence = a + b
            if len(set(sequence)) != len(sequence):
                continue
            basis = tuple(sorted(sequence))
            result[basis] = result.get(basis, sp.S.Zero) + permutation_sign(sequence) * x * y
    return simplify_form(result)

def exterior_derivative(form, de):
    terms = []
    for basis, value in form.items():
        for position, a in enumerate(basis):
            term = {(): value * (-1) ** position}
            for b in basis[:position]:
                term = wedge(term, e[b])
            term = wedge(term, de[a])
            for b in basis[position + 1:]:
                term = wedge(term, e[b])
            terms.append(term)
    return add_forms(*terms)

def component(form, *basis):
    if len(set(basis)) != len(basis):
        return sp.S.Zero
    return permutation_sign(basis) * form.get(tuple(sorted(basis)), sp.S.Zero)

def nonzero_components(entries):
    return {key: sp.factor(value) for key, value in entries.items()
            if sp.factor(value) != 0}

Forms are dictionaries of coefficients in ascending wedge order:
`{(0, 2): x}` means $x\,e^0\wedge e^2$, and `{}` means zero.
The coefficient of $e^b\wedge e^c$ with $b<c$ is the component
$T^a{}_{bc}$, since the $1/2$ cancels the two orders in the antisymmetric sum.
The same rule applies to curvature components. Scalar zero-forms use `()`.

辞書のキーは昇順の外積基底、値はその係数です。空辞書は零形式を表します。
二形式の$1/2$は反対称な二つの添字順序の和と相殺するので、昇順の基底係数が
成分値そのものです。零次形式のキーは`()`です。

## 2. Three local coframes and torsion / 三つの局所coframeとtorsion

For cyclic $(i,j,k)=(0,1,2),(1,2,0),(2,0,1)$:

$$
\begin{aligned}
S^3:&\quad de^i=\frac4s e^j\wedge e^k,\\
T^3:&\quad de^a=0,\\
\mathrm{Nil}^3:&\quad de^2=-\frac1s e^0\wedge e^1,\quad de^0=de^1=0,\\
&de^3=0,\qquad
T^i=\frac{2\eta}{s}e^j\wedge e^k-\frac V3 e^i\wedge e^3,\qquad T^3=0.
\end{aligned}
$$

These are the paper's geometric-torsion coefficients:
$T^{\mathrm{ax}}_{ijk}=2\eta\varepsilon_{ijk}/s$ and
$T^{\mathrm{vec}}_{abc}=(\delta_{ac}V_b-\delta_{ab}V_c)/3$,
with $V_a=(0,0,0,V)$. Set $V=0$ for AX or $\eta=0$ for VT.
Here $s=r$ for the sphere, $s=R_1$ for the torus ansatz, and $s=R$ for Nil.
The sphere's physical round radius is $r/2$.

上式は本文の幾何学的torsionの成分を二形式に直したものです。AXは$V=0$、
VTは$\eta=0$で得られます。$s$は球面の$r$、トーラスの$R_1$、Nilの$R$です。
球面の実半径は$r/2$であり、$s$を実半径と取り違えないでください。

In [ ]:
cyclic = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]
de_by_geometry = {
    "S3": [multiply(4/s, wedge(e[j], e[k])) for i, j, k in cyclic] + [{}],
    "T3": [{}, {}, {}, {}],
    "Nil3": [{}, {}, multiply(-1/s, wedge(e[0], e[1])), {}],
}

def torsion_ansatz(vector, axial):
    return [add_forms(multiply(2*axial/s, wedge(e[j], e[k])),
                      multiply(-vector/sp.Integer(3), wedge(e[i], e[3])))
            for i, j, k in cyclic] + [{}]

prescribed_torsion = torsion_ansatz(V, eta)
coframe_residuals = {
    name: [exterior_derivative(form, de) for form in de]
    for name, de in de_by_geometry.items()
}
coframe_residuals  # d^2 e^a = 0: each row contains four zero forms.

## 3. Solve Cartan's equation / Cartan方程式から接続を解く

We first solve for 24 metric-compatible connection coefficients.
Antisymmetry $\Gamma_{abc}=-\Gamma_{bac}$ is imposed, and Cartan's equation gives
$C^a{}_{bc}+\Gamma^a{}_{cb}-\Gamma^a{}_{bc}=T^a{}_{bc}$.
The contortion formula is not used to obtain this solution.

最初の二添字の反対称性を課し、独立な24接続係数をCartan方程式の線形系から解きます。
この接続の構成にはcontortionの閉形式を使用しません。

In [ ]:
def tensor_from_forms(forms):
    return {(a, b, c): component(forms[a], b, c)
            for a, b, c in product(indices, repeat=3)}

def solve_connection(de, torsion):
    C = tensor_from_forms(de)
    target = tensor_from_forms(torsion)
    independent = {(a, b, c): sp.Symbol(f"g_{a}{b}{c}")
                   for a, b in combinations(indices, 2) for c in indices}
    gamma = {}
    for a, b, c in product(indices, repeat=3):
        gamma[a, b, c] = (independent[a, b, c] if a < b else
                           -independent[b, a, c] if a > b else sp.S.Zero)
    equations = [C[a, b, c] + gamma[a, c, b] - gamma[a, b, c] - target[a, b, c]
                 for a in indices for b, c in combinations(indices, 2)]
    solution, = sp.solve(equations, list(independent.values()), dict=True)
    return {abc: sp.factor(value.subs(solution)) for abc, value in gamma.items()}

def connection_forms(gamma):
    return [[add_forms(*(multiply(gamma[a, b, c], e[c]) for c in indices))
             for b in indices] for a in indices]

def cartan_torsion(de, omega):
    return [add_forms(de[a], *(wedge(omega[a][b], e[b]) for b in indices))
            for a in indices]

def cartan_curvature(de, omega):
    return [[add_forms(exterior_derivative(omega[a][b], de),
                       *(wedge(omega[a][c], omega[c][b]) for c in indices))
             for b in indices] for a in indices]

## 4. Exterior-form NY calculation / 外微分形式によるNY計算

Metric compatibility and the first Bianchi identity give

$$
\begin{aligned}
d(e^a\wedge T_a)
 &=D(e^a\wedge T_a)
 =T^a\wedge T_a-e^a\wedge DT_a\\
 &=T^a\wedge T_a-e^a\wedge e^b\wedge R_{ab}.
\end{aligned}
$$

The contracted 3-form is a scalar under frame rotations, hence $D=d$ on it.
Moving a 2-form past a 1-form gives a positive sign.
This identity is general for the specified connection; its pointwise vanishing
below is a property of this static ansatz.

計量適合性と第一Bianchi恒等式から上式を得ます。縮約した3-formには$D=d$が
作用し、2-formと1-formを交換する符号は正です。恒等式自体と、以下の静的ansatzで
密度が点ごとに零になることは区別します。

Compute $T^a$, $R^a{}_b$, $Q=e^a\wedge T_a$, TT, REE and $dQ$ separately.
No NY density is entered as an expected value in this calculation.

$T^a$、$R^a{}_b$、$Q$、TT、REE、$dQ$をそれぞれ構成します。
NY密度の期待式を計算の入力にしません。

In [ ]:
cases = {}
for name, de in de_by_geometry.items():
    gamma = solve_connection(de, prescribed_torsion)
    omega = connection_forms(gamma)
    torsion = cartan_torsion(de, omega)
    curvature = cartan_curvature(de, omega)
    Q = add_forms(*(wedge(e[a], torsion[a]) for a in indices))
    TT = add_forms(*(wedge(t, t) for t in torsion))
    REE = add_forms(*(wedge(wedge(e[a], e[b]), curvature[a][b])
                      for a, b in product(indices, repeat=2)))
    dQ = exterior_derivative(Q, de)
    FULL = add_forms(TT, multiply(-1, REE))
    cases[name] = dict(de=de, gamma=gamma, omega=omega, torsion=torsion,
                       curvature=curvature, Q=Q, TT=TT, REE=REE, dQ=dQ, FULL=FULL)

# Row order: S3, T3, Nil3. Columns: n_TT, n_REE, n_FULL, coefficient of dQ.
density_table = sp.Matrix([
    [component(case[key], *volume_indices) for key in ("TT", "REE", "FULL", "dQ")]
    for case in cases.values()
])
density_table

For each geometry the result is

$$
Q=\frac{6\eta}{s}e^0\wedge e^1\wedge e^2,\qquad
n_{\mathrm{TT}}=n_{\mathrm{REE}}=-\frac{4V\eta}{s},\qquad
n_{\mathrm{FULL}}=dQ/\mathrm{vol}_4=0.
$$

One can also read this result directly from the torsion forms.
Each of the three spatial TT terms contributes $-4V\eta/(3s)$.
In $Q$, the vector terms contain a repeated basis form, leaving the three axial
terms. Each coframe differential in $d(e^0\wedge e^1\wedge e^2)$ also produces
a repeated spatial basis form, so $dQ=0$.

三つの空間TT項はそれぞれ$-4V\eta/(3s)$を与えます。$Q$ではvector項が基底の重複に
より消え、三つのaxial項が残ります。$d(e^0\wedge e^1\wedge e^2)$も、各基底の
外微分を代入すると空間基底が重複するため零です。これにより結果を数式からも追えます。

In [ ]:
form_residuals = {}
for name, case in cases.items():
    de, omega, T, R = (case[key] for key in ("de", "omega", "torsion", "curvature"))
    bianchi = [add_forms(exterior_derivative(T[a], de),
                        *(wedge(omega[a][b], T[b]) for b in indices),
                        *(multiply(-1, wedge(R[a][b], e[b])) for b in indices))
               for a in indices]
    form_residuals[name] = {
        "Cartan torsion minus prescribed torsion":
            [add_forms(T[a], multiply(-1, prescribed_torsion[a])) for a in indices],
        "first Bianchi": bianchi,
        "dQ minus TT plus REE": add_forms(case["dQ"], multiply(-1, case["TT"]), case["REE"]),
    }
form_residuals  # Every displayed residual is a zero form.

## 5. Component coefficients / 成分表示の係数

Expanding the 2-forms fixes the full-index contractions:

$$
n_{\mathrm{TT}}=\frac14\sum_{a,b,c,d,f}\varepsilon^{abcd}T^f{}_{ab}T_{fcd},
\qquad
n_{\mathrm{REE}}=\frac12\sum_{a,b,c,d}\varepsilon^{abcd}R_{abcd}.
$$

There are two factors of $1/2$ in TT and one in REE.
The code below sums all nonzero epsilon entries (the 24 permutations); terms
with repeated indices vanish. It does not restrict the sum to independent pairs.

TTには二形式の$1/2$が二つ、REEには一つあります。以下ではepsilonが非零になる
24置換を全て足します。独立な添字対だけの和ではありません。

In [ ]:
component_densities = {}
for name, case in cases.items():
    T, R = case["torsion"], case["curvature"]
    tt_sum = sum(sp.LeviCivita(a, b, c, d)
                 * sum(component(T[f], a, b) * component(T[f], c, d) for f in indices)
                 for a, b, c, d in permutations(indices))
    ree_sum = sum(sp.LeviCivita(a, b, c, d) * component(R[a][b], c, d)
                  for a, b, c, d in permutations(indices))
    component_densities[name] = (sp.factor(tt_sum / 4), sp.factor(ree_sum / 2))

# These differences vanish: component contraction minus independent wedge expansion.
sp.Matrix([[sp.factor(component_densities[name][i]
                      - component(case[key], *volume_indices))
            for i, key in enumerate(("TT", "REE"))]
           for name, case in cases.items()])

## 6. The T/τ dictionary / T/τ辞書

The paper's formulas are

$$
\begin{aligned}
\Gamma^{\mathrm{LC}}_{abc}&=\tfrac12(C_{abc}+C_{cba}-C_{bac}),\\
K_{abc}[T]&=-\tfrac12(T_{abc}+T_{bca}-T_{cab}),\\
K^{\mathrm{code}}_{abc}[\tau]&=+\tfrac12(\tau_{abc}+\tau_{bca}-\tau_{cab}).
\end{aligned}
$$

Thus $\tau=-T$, $V_\tau=-V$, $\eta_\tau=-\eta$ must give the same connection.
We compare both expressions with the connection solved directly in section 3.
We also reconstruct torsion with arbitrary input parameters $V_\tau,\eta_\tau$,
before substituting their relation to geometric parameters.

本文のcontortion式とコード入力の式を、節3で直接解いた接続に照合します。
さらに、独立した入力$V_\tau,\eta_\tau$のままCartan方程式でtorsionを再構成し、
それが入力配列の逆符号であることを確かめます。

In [ ]:
V_tau, eta_tau = sp.symbols("V_tau eta_tau", real=True)
tau_forms = torsion_ansatz(V_tau, eta_tau)
tau_tensor = tensor_from_forms(tau_forms)
T_tensor = tensor_from_forms(prescribed_torsion)
parameter_map = {V_tau: -V, eta_tau: -eta}
dictionary_residuals = {}

for name, case in cases.items():
    C = tensor_from_forms(case["de"])
    gamma_lc, gamma_T, gamma_tau = {}, {}, {}
    for a, b, c in product(indices, repeat=3):
        lc = (C[a,b,c] + C[c,b,a] - C[b,a,c]) / 2
        gamma_lc[a,b,c] = lc
        gamma_T[a,b,c] = lc - (T_tensor[a,b,c] + T_tensor[b,c,a] - T_tensor[c,a,b]) / 2
        gamma_tau[a,b,c] = lc + (tau_tensor[a,b,c] + tau_tensor[b,c,a] - tau_tensor[c,a,b]) / 2
    T_from_tau = cartan_torsion(case["de"], connection_forms(gamma_tau))
    dictionary_residuals[name] = {
        "Gamma[T] minus Cartan solution":
            nonzero_components({abc: gamma_T[abc] - case["gamma"][abc] for abc in gamma_T}),
        "Gamma_code[-T] minus Cartan solution":
            nonzero_components({abc: gamma_tau[abc].subs(parameter_map) - case["gamma"][abc]
                                for abc in gamma_tau}),
        "T reconstructed from code input plus tau":
            [add_forms(T_from_tau[a], tau_forms[a]) for a in indices],
    }
    case["gamma_lc"], case["gamma_tau"] = gamma_lc, gamma_tau
dictionary_residuals  # Empty dictionaries denote exact zero residuals.

The simultaneous sign change leaves $V^2,\eta^2,V\eta$ invariant.
It preserves the connection and therefore curvature and actions constructed
with the same NY coefficients and global normalization. It does not change
the definitions of the density coefficients or the integration domain.

同時符号反転では$V^2,\eta^2,V\eta$が不変です。同じ接続なので、同じNY係数と
大域正規化を用いる曲率・作用も同じになります。NYの係数や積分域を符号辞書で
置き換える操作ではありません。

Input intervals reflect and reverse their endpoints:

$$
V_\tau\in[v_-,v_+]\ \Longleftrightarrow\ V\in[-v_+,-v_-],\qquad
\eta_\tau\in[p,q]\ \Longleftrightarrow\ \eta\in[-q,-p].
$$

For example, the input rectangle $[0,5]\times[-10,5]$ corresponds to
$[-5,0]\times[-5,10]$ in geometric $(V,\eta)$. Plots ordered by ascending
coordinates must reorder the data as well as relabel the axes.

例えば入力域$[0,5]\times[-10,5]$は幾何学的$(V,\eta)$では
$[-5,0]\times[-5,10]$です。軸を昇順に表示する場合はデータの対応順も反転します。

## 7. Exact example and limits / 厳密な数値例と適用範囲

At geometric $s=1,V=-4,\eta=3$, the common TT and positive-REE density is
$48$, while FULL and $dQ$ vanish. The matching input is $V_\tau=4,\eta_\tau=-3$.
AX and VT are obtained by substitution, not by separate code paths.

幾何学的入力$s=1,V=-4,\eta=3$ではTTと正符号REEが$48$、FULLと$dQ$が零です。
対応するコード入力は$V_\tau=4,\eta_\tau=-3$です。AX・VTは同じ式への代入で得ます。

In [ ]:
exact_examples = {
    "MX: s=1, V=-4, eta=3": density_table.subs({s: 1, V: -4, eta: 3}),
    "AX: V=0": density_table.subs(V, 0),
    "VT: eta=0": density_table.subs(eta, 0),
}
exact_examples

This is a local exterior-algebra calculation at fixed frame-constant parameters.
It does not perform a radius scan or test the full field equations. The general
NY identity does not imply pointwise vanishing for arbitrary fields. The latter
result here uses the stated static homogeneous ansatz.

Global integrals use the paper's separate geometry: sphere radius $r/2$ and
volume $\pi^2 Lr^3/4$; torus periods $2\pi$ and volume $(2\pi)^3LR_1R_2R_3$;
Nil lattice $(2\pi,2\pi,4\pi^2)$ and volume $(2\pi)^4LR^3$.
The circle coordinate has period 1 and circumference $L$.
These domains are specified in Section 3, not derived by the local calculation.

このノートは固定したframe定数のもとでの局所外積計算です。半径走査や全場方程式の
検査ではありません。一般のNY恒等式から任意の場に対する密度の消失は導かれず、
ここでの消失は指定した静的一様ansatzによります。大域積分には第3章の球面半径・
トーラス周期・Nil格子・円周長規約を別途用います。それらを局所計算から導出したとは
扱いません。